# 07 · Mapas de ativação: para onde a rede olhou

Aulas **"Mapa de ativação: entendendo a auditoria"** e **"o que ainda não é"**: o mapa mostra **onde** a
decisão se apoiou, não **por quê**, e não é explicação. Mesmo assim, é a melhor ferramenta para pegar
um atalho no flagra.

**Grad-CAM** (Selvaraju et al., 2017): os mapas da última convolução, ponderados pelo quanto cada um
empurra a probabilidade de cardiomegalia para cima. Neste notebook:

1. Grad-CAM das redes do notebook 05 em acertos e erros do teste;
2. **quanto da atenção cai no coração**, nos pulmões e fora do tórax, no teste inteiro, com as mesmas
   caixas fixas do notebook 06;
3. o que isso diz junto com o experimento do coração tapado.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.model import avaliacao, cnn, gradcam
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import ict

configurar_estilo()
SECAO = "07-mapas-de-ativacao"

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
y = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()
tr, va, te = (conjunto == "treino"), (conjunto == "validacao"), (conjunto == "teste")
predicoes = pd.read_csv(config.DATA_PROCESSED / "predicoes.csv", sep=";").set_index("id").loc[catalogo["id"]]

ARQUIVOS = {
    "CNN pequena · cru": ("cnn_pequena_crua.keras", False),
    "CNN pequena · pré-processado": ("cnn_pequena_pre.keras", True),
    "MobileNetV2 · ajuste fino": ("mobilenet_ajuste_fino.keras", False),
}
modelos = {nome: keras.models.load_model(config.MODELS / arq) for nome, (arq, _) in ARQUIVOS.items()}
X = {False: cnn.tensores(imagens), True: cnn.tensores(imagens, preprocessar=True)}

medidas = [ict.medir_ict(im, **config.PARAMS_ICT) for im in imagens[tr]]
CAIXA_CORACAO, CAIXA_TORAX = ict.caixas_de_referencia(medidas)
def mascara(caixa):
    m = np.zeros((config.TAMANHO, config.TAMANHO), bool)
    x0, y0, x1, y1 = caixa
    m[y0:y1, x0:x1] = True
    return m
REGIOES = {"coração": mascara(CAIXA_CORACAO)}
REGIOES["tórax sem o coração"] = mascara(CAIXA_TORAX) & ~REGIOES["coração"]
REGIOES["fora do tórax"] = ~mascara(CAIXA_TORAX)
print("Caixa do coração:", CAIXA_CORACAO, "| caixa do tórax:", CAIXA_TORAX)
print({k: f"{v.mean():.1%} da imagem" for k, v in REGIOES.items()})

## 1. Grad-CAM em acertos e erros

Para cada rede, quatro grupos do teste, com o limiar de Youden escolhido na validação: verdadeiros
positivos (cardiomegalia detectada), verdadeiros negativos, falsos positivos e falsos negativos. Os
mais confiantes de cada grupo. Em vermelho, onde a rede olhou **para tomar a decisão que tomou**
(Grad-CAM da classe prevista: da saída p onde previu cardiomegalia, de 1 − p onde previu normal); a
caixa tracejada é a do coração.

In [ ]:
idx_te = np.where(te)[0]
mapas_teste, grupos_por_rede, previstos = {}, {}, {}
for nome, modelo in modelos.items():
    pre = ARQUIVOS[nome][1]
    p = predicoes[nome].to_numpy()
    limiar = avaliacao.limiar_youden(y[va], p[va])
    previsto = p[idx_te] >= limiar
    previstos[nome] = previsto
    # Explica a decisão que a rede tomou: "cardiomegalia" onde previu cardiomegalia,
    # "normal" onde previu normal.
    mapas_teste[nome] = gradcam.gradcam(modelo, X[pre][idx_te], classe=previsto.astype(int))
    real = y[idx_te] == 1
    confianca = np.abs(p[idx_te] - limiar)
    grupos_por_rede[nome] = {
        "verdadeiro positivo": np.where(previsto & real)[0],
        "verdadeiro negativo": np.where(~previsto & ~real)[0],
        "falso positivo": np.where(previsto & ~real)[0],
        "falso negativo": np.where(~previsto & real)[0],
    }
    fig, eixos = plt.subplots(4, 4, figsize=(11, 11.5))
    for linha, (grupo, sel) in enumerate(grupos_por_rede[nome].items()):
        escolhidos = sel[np.argsort(-confianca[sel])][:4]
        for col in range(4):
            ax = eixos[linha, col]
            ax.axis("off")
            if col >= len(escolhidos):
                continue
            k = escolhidos[col]
            ax.imshow(imagens[idx_te[k]], cmap="gray")
            ax.imshow(mapas_teste[nome][k], cmap="jet", alpha=0.4, vmin=0, vmax=1)
            x0, y0, x1, y1 = CAIXA_CORACAO
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="white", ls="--", lw=1))
            ax.set_title(f"{catalogo['id'][idx_te[k]]} · p={p[idx_te[k]]:.2f}", fontsize=7)
        eixos[linha, 0].text(-8, 64, grupo, rotation=90, ha="right", va="center", fontsize=10,
                             fontweight="bold", color=config.AZUL_PROFUNDO)
    fig.suptitle(f"Grad-CAM · {nome}", fontweight="bold", color=config.AZUL_PROFUNDO)
    salvar_figura(fig, SECAO, f"gradcam-{nome.split(' · ')[0].split()[0].lower()}-{'pre' if ARQUIVOS[nome][1] else 'cru'}")
    plt.show()

## 2. Quanto da atenção cai no coração

Para cada exame de teste, a fração da soma do mapa que cai em cada região. Se o mapa fosse uniforme,
a fração seria igual à **área** da região; a coluna `vezes_a_area` diz quantas vezes mais (ou menos)
atenção a região recebeu do que a área dela justificaria. Mapas vazios (nenhum gradiente positivo)
não têm atenção para repartir e ficam fora das médias; a tabela diz quantos foram.

In [ ]:
linhas = []
for nome, mapas in mapas_teste.items():
    for regiao, m in REGIOES.items():
        fr = gradcam.fracao_na_regiao(mapas, m)
        for k_classe, rot in config.ROTULOS.items():
            sel = y[idx_te] == k_classe
            linhas.append({"modelo": nome, "regiao": regiao, "classe": rot,
                           "fracao_da_atencao": np.nanmean(fr[sel]), "area": m.mean(),
                           "vezes_a_area": np.nanmean(fr[sel]) / m.mean(),
                           "mapas_vazios": float(np.isnan(fr[sel]).mean())})
atencao = pd.DataFrame(linhas)
tabela = atencao.pivot_table(index=["modelo", "regiao"], columns="classe", values="vezes_a_area")
salvar_tabela(atencao, SECAO, "atencao-por-regiao", indice=False)

fig, eixos = plt.subplots(1, len(mapas_teste), figsize=(15, 3.8), sharey=True)
for ax, (nome, mapas) in zip(eixos, mapas_teste.items()):
    media = mapas.mean(axis=0)
    ax.imshow(imagens[idx_te].mean(axis=0), cmap="gray")
    ax.imshow(media, cmap="jet", alpha=0.45)
    for caixa, cor in [(CAIXA_CORACAO, "white"), (CAIXA_TORAX, "yellow")]:
        x0, y0, x1, y1 = caixa
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=cor, ls="--", lw=1))
    ax.set_title(f"Grad-CAM médio · {nome}", fontsize=9)
    ax.axis("off")
salvar_figura(fig, SECAO, "gradcam-medio", tabela.reset_index())
plt.show()
tabela.round(2)

## Conclusões

* **Quando diz "cardiomegalia", a CNN olha a borda esquerda do coração:** nos verdadeiros positivos,
  o mapa acende na borda lateral esquerda da silhueta cardíaca (à direita na imagem), onde o
  ventrículo esquerdo aumentado empurra a silhueta para fora. É o sinal que o médico procura. Os
  falsos positivos têm o mesmo desenho: a rede viu uma borda esquerda proeminente.
* **Quando diz "normal", olha a base dos pulmões:** os mapas acendem nos seios costofrênicos e na
  cúpula do diafragma, e às vezes na borda de cima do filme, que não é anatomia.
* **A região do coração recebe mais atenção que a área justifica:** na CNN crua, 29 % da atenção dos
  exames de cardiomegalia cai na caixa do coração, que ocupa 18 % da imagem (**1,6×**); na MobileNet,
  1,9×. Mesmo assim, cerca de 40 % da atenção cai fora do tórax.
* **Um ponto quente fora do paciente:** no mapa médio das CNNs pequenas há um foco no topo central da
  imagem, acima do tórax, onde ficam etiquetas e a borda do filme. É um atalho candidato, coerente com
  o notebook 06: tapar o coração tira pouca AUC.
* **O mapa é grosso na MobileNet:** a 128 px a última camada dela tem 4×4 posições, e o Grad-CAM só
  consegue apontar uma mancha no centro. A CNN pequena, com mapas de 32×32, mostra bem mais.

**O que o mapa não é:** ele mostra **onde** a decisão se apoiou, não **por quê**. Um mapa na borda do
coração é compatível com o critério clínico, mas não prova que a rede mede o coração como o ICT mede.

> **Nota de método.** Uma primeira versão deste notebook calculava o gradiente em relação à imagem de
> entrada, e não aos mapas da `conv3`: a camada de aumento de dados, um modelo aninhado no Keras, era
> confundida com a base. O erro foi corrigido e há um teste que confere a resolução dos mapas usados
> (32×32 na CNN pequena, 4×4 na MobileNet). O mapa também passou a explicar a **classe prevista**:
> explicar "cardiomegalia" num exame que a rede acha normal dava mapas vazios.